[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/geraldmc/irilab2026/blob/main/notebooks/r2/r2-q3/02_targeted_augmentation.ipynb)

# R2-Q3 Week 3 — Train the targeted augmentation classifier

### This notebook produces `targeted_classifier.pt` plus its evaluation metrics, the third condition the comparison reads next week.

This notebook trains the targeted-augmentation classifier — the one whose augmentation pipeline is built from your precommitted failure-mode-to-augmentation mapping. Same scaffold as Week 2: data load, train, evaluate on PV-internal, evaluate on PD. The PV→PD gap you measure here is what gets compared against the kitchen-sink gap from Week 2 in the Week 4 comparison. The page's Prediction is that targeted beats kitchen-sink at gap reduction, but only by a modest margin — and that a small margin is itself evidence that kitchen-sink picked up the same signal incidentally.

By the end of this notebook you will have:

- A trained targeted classifier saved as `targeted_classifier.pt`, ready for the Week 4 comparison.
- PV-internal and PD evaluation metrics saved as `targeted_metrics.parquet`, with per-category accuracy and the PV→PD gap.
- Submitted draft presentation.

In [ ]:
# Cell 0 — probe
try:
    import irilab2026 as iri
    print(f"irilab2026 {iri.__version__} already installed.")
    print("If you want to pull the latest changes, run Cell 1 below.")
except ModuleNotFoundError:
    print("irilab2026 not installed — run Cell 1 below.")

In [ ]:
# Cell 1 — install
# First run in a fresh Colab session: run this cell.
# Later runs in the same session: skip this cell.
!pip install git+https://github.com/geraldmc/irilab2026.git@main

**A note about a possible restart prompt.** If Colab shows a "RESTART SESSION" banner after the install: click it, wait for the kernel to reconnect, then continue from the next cell. The install does not need to be run again. If no banner appears, ignore this and move on.

In [ ]:
# Cell 2 — mount and setup
# Mount Drive, set up irilab2026, point to the R2-Q3 output directory.
import irilab2026 as iri
iri.setup()

OUTPUT_DIR = iri.output_dir("r2_q3")
print(f"Output directory: {OUTPUT_DIR}")

### 1) Defensive load of `precommit.json` and the Week 2 metrics (for the comparison setup)

You ended last week with two of the three classifiers trained — the no-augmentation floor and the kitchen-sink baseline — each scored on PlantVillage and PlantDoc in the five-category disease space, with its PV→PD gap recorded. This section loads two things before any training starts: the decisions you locked in NB00, and those two sets of Week-2 metrics. Loading them up front is what lets the targeted classifier you build next be driven by the mapping you committed in Week 1 and measured against last week's numbers.

Three of the six precommit fields drive this notebook:

- `experimental_design` — confirms you're running the three-condition factorial comparison these notebooks are built around.
- `failure_mode_mapping` — the failure-mode-to-augmentation mapping you locked in NB00. Section 2 turns the addressable entries in this mapping into the targeted augmentation pipeline. This mapping *is* the targeted condition.
- `class_space_remapping` — the two class-label-to-category maps (one for PlantVillage, one for PlantDoc). Sections 4 and 5 score the targeted classifier in that shared five-category space, exactly as NB01 scored the first two.

The reason to *load* those decisions rather than retype them is the whole point of having precommitted: the targeted set you train this week should be the one you reasoned about and locked in Week 1, not a fresh choice made after seeing last week's numbers. If you find yourself wanting to change a precommit value now, that is a change to your methodology — make it in NB00 and re-run, so the record stays honest.

In [ ]:
# Read back the six decisions you locked in NB00. This notebook consumes
# three of them; the load is defensive so a missing or half-written file
# fails here, with a clear message, rather than partway through training.
import json

precommit_path = OUTPUT_DIR / "precommit.json"
if not precommit_path.exists():
    raise FileNotFoundError(
        f"No precommit.json at {precommit_path}. "
        "Run NB00 (00_orient_and_precommit.ipynb) to completion first — "
        "it writes the Week-1 decisions this notebook reads."
    )

with open(precommit_path) as f:
    precommit = json.load(f)

# NB02 consumes three of the six precommit fields:
#   experimental_design  — confirms the three-condition factorial setup.
#   failure_mode_mapping — the mapping Section 2 turns into the targeted set.
#   class_space_remapping — the two class->category maps Sections 4-5 score in.
for key in ("experimental_design", "failure_mode_mapping", "class_space_remapping"):
    if precommit.get(key) is None:
        raise KeyError(
            f"precommit.json is missing '{key}'. Re-run NB00 — every field "
            "should be populated before you start Week 3."
        )

design = precommit["experimental_design"]
fmm = precommit["failure_mode_mapping"]
csr = precommit["class_space_remapping"]

# This chain (NB01-NB03) is scaffolded for the factorial design: three named
# conditions compared head to head. NB01 trained two (baseline and
# kitchen-sink); this notebook trains the third (targeted). A hold-one-out
# design would need a different training loop, so flag it rather than silently
# running the wrong shape.
if design["design"] != "factorial":
    print(
        f"WARNING: experimental_design is {design['design']!r}, not "
        "'factorial'. These notebooks train and compare the three named "
        "factorial conditions; hold-one-out needs a different loop. Check "
        "with your mentor before continuing."
    )

# Print back what drives the rest of the notebook. The addressable entries in
# the failure-mode mapping are the ones Section 2 builds augmentations for; the
# unaddressable ones are carried on the record but deliberately left alone.
n_addressed = sum(1 for e in fmm.values() if e["addressable"])
n_excluded = sum(1 for e in fmm.values() if not e["addressable"])

print("Loaded precommit.json")
print(f"  Experimental design: {design['design']} "
      f"({design['n_conditions']} conditions)")
print(f"  Failure-mode map:    {n_addressed} addressed, "
      f"{n_excluded} deliberately not")
print(f"  Class-space remap:   PV {len(csr['pv_class_to_category'])} + "
      f"PD {len(csr['pd_class_to_category'])} classes -> "
      f"{len(csr['categories_used'])} categories")

NB02 doesn't just train — it compares. The point of Week 3 is to ask whether the targeted classifier's PV→PD gap is smaller than the kitchen-sink baseline's, so this notebook needs last week's numbers in hand. NB01 wrote two files to this question's output directory:

- `baseline_metrics.parquet` — the no-augmentation floor's per-category and overall accuracy on PlantVillage and PlantDoc, with the PV→PD gap.
- `kitchen_sink_metrics.parquet` — the same for the kitchen-sink baseline.

Each file has one row per disease category plus a final `overall` row, with columns `category`, `pv_accuracy`, `pd_accuracy`, and `gap` (the PV-internal accuracy minus the PlantDoc accuracy). Some cells are blank where a category has no examples in PlantDoc's test split — `pest` is the known case — and that's expected, not a problem.

Loading them now, defensively, means a missing file stops you here with a pointer to re-run NB01, rather than failing in Section 5 after you've already spent the time training the targeted classifier.

In [ ]:
# Load the two metrics files NB01 wrote, so Section 5 can place the targeted
# gap against the floor and the kitchen-sink baseline. Defensive on both the
# file's existence and its columns: a stale or partial file from an older NB01
# run should fail here with a clear message, not silently mis-compare later.
import pandas as pd

WEEK2_METRIC_FILES = {
    "baseline":     "baseline_metrics.parquet",      # no-augmentation floor
    "kitchen_sink": "kitchen_sink_metrics.parquet",  # strong baseline to beat
}
EXPECTED_METRIC_COLS = {"category", "pv_accuracy", "pd_accuracy", "gap"}

week2_metrics = {}
for name, fname in WEEK2_METRIC_FILES.items():
    path = OUTPUT_DIR / fname
    if not path.exists():
        raise FileNotFoundError(
            f"No {fname} at {path}. Run NB01 "
            "(01_baseline_and_kitchen_sink.ipynb) to completion first — "
            "it writes the Week-2 metrics this notebook compares against."
        )
    df = pd.read_parquet(path)

    missing = EXPECTED_METRIC_COLS - set(df.columns)
    if missing:
        raise KeyError(
            f"{fname} is missing columns {sorted(missing)}. This usually means "
            "NB01 wrote an older schema — re-run NB01 to regenerate it."
        )
    if "overall" not in set(df["category"]):
        raise ValueError(
            f"{fname} has no 'overall' row. The overall PV->PD gap is the "
            "headline number Section 5 compares against — re-run NB01."
        )
    week2_metrics[name] = df

# A small read-back: the overall gap for each Week-2 condition. These are the
# two numbers the targeted classifier is measured against in Section 5 — the
# floor (how big the gap is with no augmentation) and the bar to beat (how much
# the kitchen-sink baseline already shrank it).
def overall_gap(df):
    return float(df.loc[df["category"] == "overall", "gap"].iloc[0])

print("Loaded Week-2 metrics for comparison:")
for name in WEEK2_METRIC_FILES:
    print(f"  {name:13s} overall PV->PD gap = {overall_gap(week2_metrics[name]):.3f}")

Before training, set how this run executes. There is one knob, `DEV_MODE`, and it does two things at once: which size of PlantVillage to load, and how many training epochs to run.

Leave `DEV_MODE = True` while you are getting the notebook to run end to end. It uses a small slice of PlantVillage and a short two-epoch cap, so the leaf segmentation in Section 2, the background-randomization step, and the training run all finish in a few minutes and you catch mistakes cheaply. The targeted condition has more moving parts than the first two classifiers did, so exercising the whole path on the tiny variant first is worth the habit. Switch to `DEV_MODE = False` for the real run: the full dataset and the full ten-epoch recipe, which takes tens of minutes on a GPU but produces the numbers you will report.

Whatever you choose, the targeted run uses the same dataset size and epoch count NB01 used for the first two classifiers. That is deliberate — the only difference allowed between the three conditions is the augmentation, so dataset size and epoch count are held identical across all three. Training also requires a GPU; the cell checks for one and stops early, with instructions, if it does not find it.

In [ ]:
# Run configuration — set once here; every section below reads from it.
#
# DEV_MODE is the one knob you flip while getting the notebook running:
#   True  -> the "tiny" PV variant (~50 images/class) and a 2-epoch cap.
#            Segmentation, background randomization, and training all finish
#            in a few minutes.
#   False -> the "full" PV variant (~54k images) and the full 10-epoch
#            recipe. The real run: tens of minutes on a GPU.
#
# The targeted run uses the SAME variant and epoch cap NB01 used for the
# baseline and kitchen-sink classifiers. That is on purpose: the only thing
# allowed to differ across the three conditions is the augmentation. Holding
# the data and the epoch count identical is what makes the gap comparison fair.
DEV_MODE = True

PV_VARIANT = "tiny" if DEV_MODE else "full"
EPOCH_CAP = 2 if DEV_MODE else None   # None -> train_baseline's full 10 epochs

# One seed for the whole notebook. train_baseline also seeds itself per call
# from its own `seed=` argument; seeding here keeps anything outside that call
# (data loading, segmentation sampling, background randomization, evaluation
# ordering) reproducible too.
iri.seed_all(42)

# Training calls .cuda() directly, so a GPU is not optional in this notebook.
# Fail now, with a clear message, rather than partway through training with a
# cryptic CUDA error.
assert iri.has_gpu(), (
    "No GPU detected. This notebook trains a ResNet-18 classifier and needs a "
    "GPU. In Colab: Runtime -> Change runtime type -> T4 GPU, then re-run from "
    "the top."
)

print(f"DEV_MODE   = {DEV_MODE}")
print(f"PV_VARIANT = {PV_VARIANT!r}")
print(f"EPOCH_CAP  = {EPOCH_CAP}")
print("Seed set to 42; GPU detected.")

### 2) Build the targeted augmentation pipeline.

The targeted condition is the augmentation pipeline built from your Precommit 4 mapping. It has two parts, and they are built differently.

The first part is the stock part: the leaf-shape and lighting augmentations are ordinary torchvision transforms — random crops, perspective shifts, color jitter — and they compose exactly the way last week's kitchen-sink (RandAugment) pipeline did. You build that part yourself, in the practice below.

The second part is background randomization, and it is not a stock transform. To randomize an image's background you first have to separate the leaf from it — *segment* the leaf — so you can keep the leaf and replace everything else. The next three cells handle that segmentation and the background swap. They are given, with comments, because implementing leaf segmentation is well beyond a week's work; what to take from them is *what* they do and *why*, not how to write them from scratch.

One honest caveat up front, inherited from R2-Q2's use of segmentation: the masks are a sanity-checked approximation, not ground truth. Any conclusion that leans on the background augmentation should carry that uncertainty forward.

First, load PlantVillage and segment the leaf in every training image. The segmentation uses GrabCut with a *border-is-background* prior: PlantVillage photographs put a single leaf on a near-uniform studio backdrop, so marking the outer frame of the image as definite background and the interior as probable foreground lets GrabCut separate the two by keying on the uniform backdrop. That trick is what makes a model-free method work here even on diseased leaves — where a green-pixel color rule would fail, because the leaf is brown or spotted.

Segmenting every training image is the one-time expensive step, so the result is cached: the first run segments and saves, and every run after that loads from the cache in seconds. On the tiny variant the first run is also quick; on the full variant it takes tens of minutes, once.

In [ ]:
# Load PlantVillage once, at the variant set in Section 1. The targeted
# classifier trains on the same PV data the other two conditions used; only the
# augmentation differs.
metadata, hf_dataset = iri.load_plantvillage(variant=PV_VARIANT)

# Read the class count from the data rather than hardcoding 38.
num_classes = metadata["class_idx"].nunique()

# Background randomization is applied only to training images, so the leaves we
# segment are the train split's.
train_meta = metadata[metadata["split"] == "train"]

print(f"PlantVillage ({PV_VARIANT}): {len(metadata)} images, "
      f"{len(train_meta)} in the train split, {num_classes} classes.")

In [ ]:
# Leaf segmentation (given infrastructure).
#
# segment_leaf: one image -> a {0,1} leaf mask, via GrabCut with a
#   border-is-background prior (frame = definite background, interior =
#   probable foreground). cv2 is preinstalled in Colab.
# build_or_load_masks: segment the whole train split once and cache the masks;
#   load from cache on later runs. Masks are bit-packed (np.packbits) so the
#   full set fits comfortably in memory and on disk.
import cv2
import numpy as np
from tqdm.auto import tqdm

MASK_CACHE = iri.cache_dir() / f"pv_leaf_masks_{PV_VARIANT}.npz"


def segment_leaf(pil_image, border_frac=0.06, iters=5):
    """Return a {0,1} uint8 HxW leaf mask for one RGB PIL image."""
    rgb = np.asarray(pil_image)                       # HxWx3, uint8
    h, w = rgb.shape[:2]
    bgr = cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)        # cv2 works in BGR

    gc = np.full((h, w), cv2.GC_PR_FGD, dtype=np.uint8)   # interior: probable FG
    m = max(1, int(round(border_frac * min(h, w))))
    gc[:m, :] = cv2.GC_BGD;  gc[-m:, :] = cv2.GC_BGD      # frame: definite BG
    gc[:, :m] = cv2.GC_BGD;  gc[:, -m:] = cv2.GC_BGD

    bgd_model = np.zeros((1, 65), np.float64)
    fgd_model = np.zeros((1, 65), np.float64)
    cv2.grabCut(bgr, gc, None, bgd_model, fgd_model, iters, cv2.GC_INIT_WITH_MASK)

    # GrabCut leaves four labels; foreground is the definite + probable FG.
    return np.where((gc == cv2.GC_FGD) | (gc == cv2.GC_PR_FGD), 1, 0).astype(np.uint8)


def build_or_load_masks(train_meta, hf_dataset, cache_path):
    """Segment every training leaf once and cache; load from cache thereafter.

    Returns a dict: image index -> (packed_mask, (H, W)). The image index is
    the metadata row's name (its position in hf_dataset), the same key
    PlantVillageDataset uses to look images up.
    """
    if cache_path.exists():
        data = np.load(cache_path)
        H, W = int(data["hw"][0]), int(data["hw"][1])
        masks = {int(i): (data["packed"][k], (H, W))
                 for k, i in enumerate(data["indices"])}
        print(f"Loaded {len(masks)} cached masks from {cache_path}")
        return masks

    print(f"No cache at {cache_path} — segmenting {len(train_meta)} images "
          "(one-time; cached for next run).")
    # PlantVillage images are a uniform size; take it from the first image and
    # keep every mask at that size so they stack into one array.
    first = hf_dataset[int(train_meta.iloc[0].name)]["image"].convert("RGB")
    H, W = np.asarray(first).shape[:2]

    indices, packed_rows = [], []
    for i in tqdm(range(len(train_meta))):
        idx = int(train_meta.iloc[i].name)
        img = hf_dataset[idx]["image"].convert("RGB")
        if img.size != (W, H):              # PIL size is (W, H)
            img = img.resize((W, H))
        mask = segment_leaf(img)            # {0,1} HxW
        indices.append(idx)
        packed_rows.append(np.packbits(mask.reshape(-1)))

    cache_path.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(cache_path, indices=np.array(indices),
                        packed=np.stack(packed_rows), hw=np.array([H, W]))
    masks = {idx: (packed_rows[k], (H, W)) for k, idx in enumerate(indices)}
    print(f"Segmented and cached {len(masks)} masks to {cache_path}")
    return masks


masks = build_or_load_masks(train_meta, hf_dataset, MASK_CACHE)

With a leaf mask in hand, the background swap needs something to swap *in*. The generator below produces a fresh synthetic background each time it is called — a random solid color, random RGB noise, or a two-color gradient. These are deliberately not realistic and deliberately not PlantDoc. The goal of the background augmentation is to make the background *unreliable* as a cue, so the model stops leaning on it; varied nonsense backgrounds do that. Pulling backgrounds from PlantDoc would instead leak the target domain into training, which would make the lab→field gap you measure dishonestly small.

In [ ]:
# A synthetic replacement background (given). One of three kinds, chosen at
# random: solid color, RGB noise, or a two-color horizontal gradient. Never a
# real field image — variety the model cannot key on is the point, not realism.
def random_background(h, w, rng):
    """Return an HxWx3 uint8 background; `rng` is a numpy Generator."""
    kind = rng.integers(0, 3)
    if kind == 0:                                    # solid color
        color = rng.integers(0, 256, size=3, dtype=np.uint8)
        return np.broadcast_to(color, (h, w, 3)).copy()
    if kind == 1:                                    # RGB noise
        return rng.integers(0, 256, size=(h, w, 3), dtype=np.uint8)
    # two-color horizontal gradient
    c0 = rng.integers(0, 256, size=3).astype(np.float32)
    c1 = rng.integers(0, 256, size=3).astype(np.float32)
    t = np.linspace(0.0, 1.0, w, dtype=np.float32)[None, :, None]   # (1, w, 1)
    grad = c0 * (1.0 - t) + c1 * t                                  # (1, w, 3)
    return np.broadcast_to(grad, (h, w, 3)).astype(np.uint8).copy()

Now tie the two together in a dataset. `BackgroundRandomizedPVDataset` is a small subclass of `iri.PlantVillageDataset`: for each image it reads, it cuts the leaf out using that image's cached mask, pastes it onto a freshly drawn background, and only then applies the usual transform. Because a new background is drawn every time an image is read, the same leaf is seen against many backgrounds across the training run — which is exactly what the background augmentation is for.

This is also *why* background randomization lives in a dataset rather than in the transform: a torchvision transform is handed only the image, never which image it is, so it cannot look up that image's mask. The dataset can, because it knows the row.

The subclass keeps the base class's `(metadata, hf_dataset, transform=)` constructor and adds the masks, so it can stand in anywhere a dataset class is expected — which is how Section 3 will hand it to `train_baseline`.

In [ ]:
# Background-randomizing dataset (given). Subclasses PlantVillageDataset and
# overrides __getitem__ to composite the leaf onto a random background before
# the transform runs. Same constructor as the base class plus `masks` and a
# `seed`, so it drops in wherever a PlantVillageDataset would.
from PIL import Image


class BackgroundRandomizedPVDataset(iri.PlantVillageDataset):
    def __init__(self, metadata, hf_dataset, transform=None, *, masks, seed=42):
        super().__init__(metadata, hf_dataset, transform=transform)
        self.masks = masks
        self.rng = np.random.default_rng(seed)

    def __getitem__(self, idx):
        row = self.metadata.iloc[idx]
        image = self.hf_dataset[int(row.name)]["image"].convert("RGB")
        packed, (H, W) = self.masks[int(row.name)]
        if image.size != (W, H):                         # keep image and mask aligned
            image = image.resize((W, H))
        arr = np.asarray(image)                          # HxWx3 uint8
        mask = np.unpackbits(packed)[:H * W].reshape(H, W)   # {0,1}
        bg = random_background(H, W, self.rng)
        composited = np.where(mask[:, :, None] == 1, arr, bg).astype(np.uint8)
        image = Image.fromarray(composited)
        if self.transform is not None:
            image = self.transform(image)
        return image, int(row["class_idx"])

That covers background randomization. The other two addressable failure modes — leaf-shape and lighting — are ordinary torchvision transforms, and this is where your Precommit 4 mapping turns into code.

The cell below gives you two things. `AUG_BY_NAME` is a menu: each augmentation *name* your mapping can list (`random_crop`, `perspective`, `brightness`, and so on), paired with the concrete torchvision transform that realizes it. NB00 fixed *which* augmentation counters *which* failure mode; the concrete parameters — crop scale, jitter strength — live here, in NB02, as that notebook said they would. `TARGETED_TAIL` is the fixed ending every train transform shares: it turns the augmented PIL image into the normalized `(3, 224, 224)` tensor the model expects, with the same ImageNet normalization the other two conditions use.

Notice one name is missing from the menu: `background_randomization`. That augmentation is handled by the dataset above, not by the transform, so it is deliberately absent here — which means when you walk the mapping in the practice, it gets skipped on its own.

In [ ]:
# The augmentation menu and the shared tail (given).
import torchvision.transforms as T

# ImageNet normalization — must match how the model was pretrained and how the
# other two conditions normalize, or the targeted classifier would see a
# different input distribution. Same constants iri's transforms use.
_MEAN = [0.485, 0.456, 0.406]
_STD = [0.229, 0.224, 0.225]

# Each augmentation NAME the mapping can list, realized as a concrete transform
# with this notebook's chosen parameters. "background_randomization" is NOT here
# — the mask-aware dataset handles it — so walking the mapping skips it cleanly.
AUG_BY_NAME = {
    # leaf_shape_attended: break the clean leaf outline (a host-species, and so
    # disease, shortcut). RandomResizedCrop also brings the image to 224x224.
    "random_crop": T.RandomResizedCrop(224, scale=(0.6, 1.0), antialias=True),
    "perspective": T.RandomPerspective(distortion_scale=0.3, p=0.5),
    # lighting_attended: vary PlantVillage's consistent studio lighting.
    "brightness": T.ColorJitter(brightness=0.3),
    "contrast": T.ColorJitter(contrast=0.3),
    "color_jitter": T.ColorJitter(saturation=0.3, hue=0.05),
}

# The fixed tail every train transform ends with: PIL image -> normalized
# (3, 224, 224) tensor. The augmentations run first, on the PIL image;
# RandomResizedCrop(224), when present, is what sizes the image to 224.
TARGETED_TAIL = [T.ToTensor(), T.Normalize(mean=_MEAN, std=_STD)]

### Practice 2.1 — Build the targeted transform

You have the menu (`AUG_BY_NAME`) and the fixed tail (`TARGETED_TAIL`). Build the targeted transform from your precommitted mapping. Walk `failure_mode_mapping` (the `fmm` you loaded in Section 1): for each entry that is `addressable`, take its `augmentations` list, and for every augmentation name that appears in `AUG_BY_NAME`, collect that transform. Then build `targeted_transform = T.Compose(collected + TARGETED_TAIL)`.

Two things to notice. First, `background_randomization` is addressable but is not in `AUG_BY_NAME` — the dataset handles it — so walking the menu skips it on its own; you do not special-case it. Second, the order is already right: the mapping lists leaf-shape (crop, then perspective) before lighting (color), which is the order you want — geometric before color, sizing crop first.

The sanity cell after this one reads `targeted_transform`, so if you skip this it will stop with a `NameError`.

In [ ]:
### Practice 2.1 — your code here --------------------------------------------------
#
#
#
#
#
# -----------------------------------------------------------------------------------

Before training on it, check that the whole pipeline composes: segmentation → background swap → augmentation → tensor. The cell below builds the targeted training dataset, reads one item, and confirms it comes out as a `(3, 224, 224)` tensor — catching a broken transform now, cheaply, instead of partway through a training run. It also reports the mean leaf (foreground) fraction across a sample of masks: a quick check that the masks are reasonable — a leaf should fill a fair share of the frame, not almost none or almost all of it. Like R2-Q2's IoU check, this is a sanity check, not a precision estimate.

In [ ]:
# Sanity check before training (given). Confirm the pipeline composes and the
# masks look reasonable. Both are checks, not measurements.
targeted_train_dataset = BackgroundRandomizedPVDataset(
    train_meta, hf_dataset, transform=targeted_transform, masks=masks,
)

image, label = targeted_train_dataset[0]
assert tuple(image.shape) == (3, 224, 224), (
    f"targeted_transform produced a {tuple(image.shape)} tensor, not "
    "(3, 224, 224). Check that your composed transform includes the crop that "
    "sizes images to 224 and ends with TARGETED_TAIL."
)
print(f"Pipeline OK: item 0 -> tensor {tuple(image.shape)}, label {label}.")

# Mean foreground fraction over a sample of masks — a leaf should occupy a fair
# chunk of the frame. A sanity check on segmentation quality, not a precision
# estimate; carry that caveat into any background-dependent conclusion.
sample = list(masks)[:200]
fracs = [np.unpackbits(masks[i][0])[: masks[i][1][0] * masks[i][1][1]].mean()
         for i in sample]
print(f"Mean leaf fraction over {len(sample)} masks: {np.mean(fracs):.2f}  "
      "(sanity check, not a precision estimate)")

### 3) Train the targeted classifier

This is the one training run NB02 makes. The recipe is identical to last week's two — same architecture, optimizer, epoch count, and seed — because the only thing allowed to differ across the three conditions is the augmentation. Here the augmentation is your targeted set: the background-randomizing training dataset from Section 2, with the `targeted_transform` you composed applied on top.

Two pieces wire the targeted condition into `train_baseline`. The training data comes from `BackgroundRandomizedPVDataset`, but `train_baseline` builds its dataset by calling `dataset_class(metadata, hf_dataset, transform=...)` — it knows nothing about masks. `functools.partial` binds the masks in, leaving a callable with exactly that signature. And `val_dataset_class=iri.PlantVillageDataset` tells `train_baseline` to build its validation set from the plain dataset, so the validation images keep their real backgrounds. Background randomization is a training-time augmentation; a validation set whose backgrounds change every epoch would make "best validation epoch" meaningless.

There is no practice cell here. The work that was yours — turning the precommitted mapping into the augmentation — was Section 2's practice. Once the augmentation is built, training is the same call you have made twice before.

In [ ]:
from functools import partial

# Train the targeted classifier. Same recipe as the baseline and kitchen-sink
# runs in NB01 — only the augmentation differs.
#
# dataset_class:     the background-randomizing dataset, with masks bound in via
#                    partial so it still matches the
#                    (metadata, hf_dataset, transform=) shape train_baseline calls.
# val_dataset_class: the plain PlantVillageDataset, so validation images keep
#                    their real backgrounds and "best validation epoch" stays a
#                    stable target across epochs.
# train_transform:   the targeted_transform you composed in Practice 2.1.
targeted_state_dict, targeted_history = iri.train_baseline(
    metadata,
    hf_dataset,
    dataset_class=partial(BackgroundRandomizedPVDataset, masks=masks),
    val_dataset_class=iri.PlantVillageDataset,
    num_classes=num_classes,
    train_transform=targeted_transform,
    seed=42,              # same seed as the other two conditions
    epoch_cap=EPOCH_CAP,  # 2 in DEV_MODE, None (full 10 epochs) otherwise
)

In [ ]:
# Best validation accuracy reached, on PV's held-out (clean-background)
# validation slice. A training-health check, not the headline result — the
# comparison that matters is the PV->PD gap, computed in Sections 4 and 5.
print("Targeted classifier trained.")
print(f"  Best PV-validation accuracy: {targeted_history['best_val_acc']:.4f} "
      f"(epoch {targeted_history['best_val_epoch'] + 1})")

### 4) Evaluate on PV-internal: per-class accuracy on the held-out PV split

### 5) Evaluate on PD; compute the targeted PV→PD gap

### 6) Closeout: save `targeted_classifier.pkl` and `targeted_metrics.parquet`; submit draft presentation